In [1]:
import pandas as pd
import unicodedata
import re

def normalize_name(name):
    # Strip diacritics (Şengün -> Sengun, Dončić -> Doncic)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    # Remove periods (A.J. -> AJ)
    name = name.replace('.', '')
    # Normalize suffixes: Sr., Jr., II, III, IV
    name = re.sub(r'\b(Sr|Jr|I{2,3}|IV)\b', '', name)
    # Collapse whitespace and lowercase
    name = ' '.join(name.lower().split())
    return name.strip()

In [3]:
estimated_wins_df_detailed = pd.read_csv('estimated_wins.csv')
salaries_df = pd.read_csv('player_salaries.csv')

salaries_25_26 = salaries_df[['Player', '2025-26']]
estimated_wins_df = estimated_wins_df_detailed[['player_name', 'ewins', 'team_alias', 'pos_text', 'age', 'mpg']].rename(columns={'player_name': 'Player'})

# Create normalized join keys
estimated_wins_df['join_key'] = estimated_wins_df['Player'].apply(normalize_name)
salaries_25_26 = salaries_25_26.copy()
salaries_25_26['join_key'] = salaries_25_26['Player'].apply(normalize_name)

In [4]:
combined_df = pd.merge(estimated_wins_df, salaries_25_26.drop(columns=['Player']), on='join_key', how='inner')
combined_df['2025-26'] = combined_df['2025-26'].replace({'\$': '', ',': ''}, regex=True).astype(int)

combined_df['production_value'] = combined_df['ewins'] * 3_800_000
combined_df['diff'] = combined_df['production_value'] - combined_df['2025-26']

combined_df = combined_df.drop(columns=['join_key'])

In [5]:
combined_df.sort_values(by=['diff'], ascending=False).to_csv('production_value_difference.csv', index=False)

In [6]:
# Top 10 best value contracts (most production above salary)
top_value = combined_df.nlargest(10, 'diff')[['Player', 'team_alias', 'pos_text', 'ewins', '2025-26', 'production_value', 'diff']]
print("=== TOP 10 BEST VALUE CONTRACTS ===")
print(top_value.to_string(index=False))
top_value.to_csv('top_10_best_value.csv', index=False)

# Top 10 most overpaid contracts
worst_value = combined_df.nsmallest(10, 'diff')[['Player', 'team_alias', 'pos_text', 'ewins', '2025-26', 'production_value', 'diff']]
print("\n=== TOP 10 MOST OVERPAID CONTRACTS ===")
print(worst_value.to_string(index=False))
worst_value.to_csv('top_10_most_overpaid.csv', index=False)

=== TOP 10 BEST VALUE CONTRACTS ===
                 Player team_alias pos_text    ewins  2025-26  production_value       diff
      Victor Wembanyama        SAS        C 13.45050 13376880        51111900.0 37735020.0
            Jalen Duren        DET        C  9.60324  6483144        36492312.0 30009168.0
          Amen Thompson        HOU       SG  9.88842  9690600        37575996.0 27885396.0
Shai Gilgeous-Alexander        OKC       PG 17.15280 38333050        65180640.0 26847590.0
          Chet Holmgren        OKC        C 10.66520 13731368        40527760.0 26796392.0
       Collin Gillespie        PHX       PG  7.64644  2296274        29056472.0 26760198.0
           Kon Knueppel        CHA       SF  9.61673 10015680        36543574.0 26527894.0
          Neemias Queta        BOS        C  7.18971  2349578        27320898.0 24971320.0
          Dyson Daniels        ATL       SG  8.44467  7707709        32089746.0 24382037.0
        Donovan Clingan        POR        C  7.89379  

In [7]:
# Team-level analysis: total surplus value (production_value - salary) per team
team_surplus = combined_df.groupby('team_alias').agg(
    total_salary=('2025-26', 'sum'),
    total_production_value=('production_value', 'sum'),
    total_surplus=('diff', 'sum'),
    num_players=('Player', 'count')
).sort_values('total_surplus', ascending=False)

print("=== TEAM SURPLUS VALUE (production value - salary) ===")
print(team_surplus.to_string())
team_surplus.to_csv('team_surplus.csv')

=== TEAM SURPLUS VALUE (production value - salary) ===
            total_salary  total_production_value  total_surplus  num_players
team_alias                                                                  
OKC            179773002            2.355840e+08   5.581096e+07           14
SAS            184204928            2.105708e+08   2.636582e+07           17
DET            169872812            1.928948e+08   2.302198e+07           14
BOS            189348982            1.970023e+08   7.653328e+06           13
HOU            171960146            1.752870e+08   3.326824e+06           14
MIA            159804115            1.624445e+08   2.640424e+06           14
CHA            179622853            1.732489e+08  -6.373975e+06           17
POR            154036459            1.348788e+08  -1.915765e+07           15
DEN            195921026            1.718328e+08  -2.408826e+07           16
TOR            186565404            1.593346e+08  -2.723082e+07           15
ATL            183728

In [8]:
# Cost per estimated win by position
pos_analysis = combined_df.groupby('pos_text').agg(
    avg_salary=('2025-26', 'mean'),
    avg_ewins=('ewins', 'mean'),
    avg_diff=('diff', 'mean'),
    num_players=('Player', 'count')
).sort_values('avg_diff', ascending=False)

pos_analysis['cost_per_ewin'] = (pos_analysis['avg_salary'] / pos_analysis['avg_ewins']).astype(int)

print("=== PRODUCTION VALUE BY POSITION ===")
print(pos_analysis.to_string())
pos_analysis.to_csv('value_by_position.csv')

=== PRODUCTION VALUE BY POSITION ===
            avg_salary  avg_ewins      avg_diff  num_players  cost_per_ewin
pos_text                                                                   
SF        9.578344e+06   1.958218 -2.137117e+06          101        4891358
C         1.109191e+07   2.352009 -2.154271e+06          106        4715927
SG        1.066630e+07   2.239234 -2.157217e+06          104        4763372
PG        1.389408e+07   2.848793 -3.068665e+06           76        4877180
PF        1.320847e+07   1.956611 -5.773352e+06           93        6750690


In [9]:
# Age curve: are younger or older players a better value?
combined_df['age_group'] = pd.cut(combined_df['age'], bins=[18, 22, 25, 28, 31, 35, 45],
                                   labels=['19-22', '23-25', '26-28', '29-31', '32-35', '36+'])

age_analysis = combined_df.groupby('age_group', observed=True).agg(
    avg_salary=('2025-26', 'mean'),
    avg_ewins=('ewins', 'mean'),
    avg_diff=('diff', 'mean'),
    num_players=('Player', 'count')
)

age_analysis['cost_per_ewin'] = (age_analysis['avg_salary'] / age_analysis['avg_ewins']).astype(int)

print("=== VALUE BY AGE GROUP ===")
print(age_analysis.to_string())
age_analysis.to_csv('value_by_age.csv')

=== VALUE BY AGE GROUP ===
             avg_salary  avg_ewins      avg_diff  num_players  cost_per_ewin
age_group                                                                   
19-22      5.196256e+06   1.489450  4.636527e+05          101        3488708
23-25      8.601006e+06   2.110533 -5.809813e+05          143        4075277
26-28      1.476108e+07   2.979411 -3.439320e+06          100        4954362
29-31      1.783903e+07   2.850145 -7.008478e+06           70        6258989
32-35      1.536588e+07   1.705452 -8.885162e+06           46        9009855
36+        1.751759e+07   2.515181 -7.959907e+06           20        6964745


In [10]:
# Best value per team: each team's most underpaid player
best_per_team = combined_df.loc[combined_df.groupby('team_alias')['diff'].idxmax()]
best_per_team = best_per_team[['Player', 'team_alias', 'ewins', '2025-26', 'diff']].sort_values('diff', ascending=False)

print("=== BEST VALUE PLAYER PER TEAM ===")
print(best_per_team.to_string(index=False))
best_per_team.to_csv('best_value_per_team.csv', index=False)

=== BEST VALUE PLAYER PER TEAM ===
                 Player team_alias     ewins  2025-26       diff
      Victor Wembanyama        SAS 13.450500 13376880 37735020.0
            Jalen Duren        DET  9.603240  6483144 30009168.0
          Amen Thompson        HOU  9.888420  9690600 27885396.0
Shai Gilgeous-Alexander        OKC 17.152800 38333050 26847590.0
       Collin Gillespie        PHX  7.646440  2296274 26760198.0
           Kon Knueppel        CHA  9.616730 10015680 26527894.0
          Neemias Queta        BOS  7.189710  2349578 24971320.0
          Dyson Daniels        ATL  8.444670  7707709 24382037.0
        Donovan Clingan        POR  7.893790  7178400 22818002.0
           Ryan Rollins        MIL  5.793950  4000000 18017010.0
       Donte DiVincenzo        MIN  7.632410 11990000 17013158.0
  Sandro Mamukelashvili        TOR  4.983260  2461463 16474925.0
          Austin Reaves        LAL  7.228300 13937574 13529966.0
             Saddiq Bey        NOP  5.122390  6118644 1

In [11]:
# Perform outer merge with indicator column
merged_df = pd.merge(estimated_wins_df, salaries_25_26.drop(columns=['Player']), on='join_key', how='outer', indicator=True)

# Find players missing from salaries_25_26 (present in estimated_wins_df but not in salaries)
missing_in_salaries = merged_df[merged_df['_merge'] == 'left_only']['Player']

# Find players missing from estimated_wins_df (present in salaries but not in estimated_wins_df)
missing_in_wins = merged_df[merged_df['_merge'] == 'right_only']['join_key']

# Print results
print("Players missing from salary data:")
print(missing_in_salaries.to_list())

print("\nPlayers missing from estimated wins data:")
print(missing_in_wins.to_list())

Players missing from salary data:
['A.J. Lawson', 'Alijah Martin', 'Andersson Garcia', 'Antonio Reeves', 'Bez Mbeng', 'Blake Hinson', 'Branden Carlson', 'Brooks Barnhizer', 'Caleb Love', 'Chaney Johnson', 'Chris Youngblood', 'Cody Martin', 'Daeqwon Plowden', 'DaQuan Jeffries', 'David Jones Garcia', 'DeJon Jarreau', 'Drew Peterson', 'Dylan Cardwell', 'Egor Demin', 'Elijah Harkless', 'Ethan Thompson', 'Isaiah Crawford', 'Isaiah Livers', 'Jahmai Mashack', 'Jahmir Young', 'Jalen Slawson', 'Jamal Cain', 'Jamaree Bouyea', 'Jamir Watkins', 'Javon Small', 'Javonte Cooke', 'John Poulakidas', 'Johnny Juzang', 'Julian Reese', 'Kennedy Chandler', 'Kevin McCullar Jr.', 'KJ Simpson', 'Lachlan Olbrich', 'Leaky Black', 'LJ Cryer', 'Luke Travers', 'Malevy Leons', 'MarJon Beauchamp', 'Miles Kelly', 'Moussa Cisse', 'Nate Williams', 'Omer Yurtseven', 'Oscar Tshiebwe', 'Pete Nance', 'PJ Hall', 'Quenton Jackson', 'RayJ Dennis', 'Ron Harper Jr.', 'Ronald Holland II', 'Ryan Nembhard', 'Sharife Cooper', 'Taelo

In [12]:
# Combine both lists into one DataFrame
missing_players_df = pd.DataFrame({'Player': pd.concat([missing_in_salaries, missing_in_wins]).unique()})

# Save to Excel
missing_players_df.to_excel('missing_players.xlsx', index=False)

# Display the first few rows
print(missing_players_df.head())

             Player
0       A.J. Lawson
1     Alijah Martin
2  Andersson Garcia
3    Antonio Reeves
4         Bez Mbeng
